In [ ]:
import sys
import os
import random
import torch
import numpy as np
import torchvision.transforms as transforms
from PIL import Image
from pathlib import Path
import wandb

# %matplotlib widget
from matplotlib import pyplot as plt

In [ ]:
sys.path.append(os.path.abspath("../"))
sys.path.append(os.path.abspath("."))

from importlib import reload

import Segmentation.PostProcessing.segmentation_postprocessing
import Segmentation.SAM.sam_lora
import Segmentation.SAM.sam_lora_util
import Environment.env_utils
import Segmentation.PreProcessing.dataset_util

import JunctionDetection.SkeletonizeDetect.segmentation_junction_detection

import evaluation_util

reload(Segmentation.PostProcessing.segmentation_postprocessing)
reload(Segmentation.SAM.sam_lora)
reload(Segmentation.SAM.sam_lora_util)
reload(Environment.env_utils)
reload(Segmentation.PreProcessing.dataset_util)
reload(JunctionDetection.SkeletonizeDetect.segmentation_junction_detection)
reload(evaluation_util)

from evaluation_util import (
    format_score,
    load_transform_image,
    get_betti_at_thresholds,
    plot_betti_curve,
)

In [ ]:
Environment.env_utils.load_segmentation_env()

SEED = Environment.env_utils.load_as("SEED", int, 42)

RAW_DATA_DIR = os.getenv("RAW_DATA_DIR")
MODEL_CHECKPOINTS_DIR = os.getenv("MODEL_CHECKPOINTS_DIR")
DATASETS_DIR = os.getenv("DATASETS_DIR")

HIGHRES_IMG_PATCHES_DIR_NAME = os.getenv(
    "HIGHRES_IMG_PATCHES_DIR_NAME", "img_patches_1024")
HIGHRES_MASK_PATCHES_DIR_NAME = os.getenv(
    "HIGHRES_MASK_PATCHES_DIR_NAME", "mask_patches_1024")
HIGHRES_IMG_DIR_NAME = os.getenv("HIGHRES_IMG_DIR_NAME", "images_4096")
HIGHRES_MASK_DIR_NAME = os.getenv("HIGHRES_MASK_DIR_NAME", "masks_4096")

WANDB_ENTITY = os.getenv("WANDB_ENTITY", "EM_IMCR_BIOVSION")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "ForkSight-SAM")

DATASET_JUNCTION_COORDS_CVAT_XML_PATH = os.getenv(
    "DATASET_JUNCTION_COORDS_CVAT_XML_PATH", None)

POSTPROCESSING_MIN_OBJ_SIZE = Environment.env_utils.load_as(
    "POSTPROCESSING_MIN_OBJ_SIZE", int, 100)

if RAW_DATA_DIR is None or MODEL_CHECKPOINTS_DIR is None or DATASETS_DIR is None:
    raise ValueError(
        "RAW_DATA_DIR, MODEL_CHECKPOINTS_DIR or DATASETS_DIR environment variable not set.")

SAVE_PLOTS = False
IGNORE_SWEEP_RUNS = True

In [ ]:
loss_config_keys = ["bce_loss_weight", "cl_dice_loss_weight", "dice_loss_weight", "focal_loss_weight",
                    "junction_heatmap_weight_scale", "junction_patch_weight", "skeleton_recall_loss_weight", "topological_loss_weight"]

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
print("cuda available: ", torch.cuda.is_available())
!nvidia-smi --query-gpu=index,utilization.gpu --format=csv

device_idx = int(input("Enter the index of the cuda device: "))
device = torch.device(f"cuda:{device_idx}" if torch.cuda.is_available() else "cpu")

print(f"\nUsing device: {device}")

In [ ]:
runs = [run for run in list(wandb.Api().runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}")) if
        Segmentation.SAM.sam_lora_util.EVALUATED_TAG in run.tags
        and run.state == "finished"
        and (run.sweep_name is None or run.sweep_name == "" or not IGNORE_SWEEP_RUNS)]

for i, run in enumerate(runs):
    used_loss_terms = [f"{key} = {run.config[key]}"
                       for key in loss_config_keys if key in run.config and float(run.config[key]) != 0.0]
    print(f"({i}) {run.name}")
    if len(used_loss_terms) > 0:
        print("        " + ", ".join(used_loss_terms))

run_idx = int(input("Enter the index of the run to evaluate: "))
run = runs[run_idx]
print(f"\nSelected run: {run.name} ({run.id})")

DOWNSAMPLE_SIZE = run.config.get("dataset_downsample_size", None)
if DOWNSAMPLE_SIZE is not None and isinstance(DOWNSAMPLE_SIZE, (list, tuple)) and len(DOWNSAMPLE_SIZE) == 2:
    DOWNSAMPLE_SIZE = (DOWNSAMPLE_SIZE[0], DOWNSAMPLE_SIZE[1]) if isinstance(
        DOWNSAMPLE_SIZE, list) else DOWNSAMPLE_SIZE
    print(f"Downsample size: {DOWNSAMPLE_SIZE}")
else:
    DOWNSAMPLE_SIZE = None

In [ ]:
model_param_artifacts = [a for a in list(
    run.logged_artifacts()) if a.type == "model"]
for i, artifact in enumerate(model_param_artifacts):
    print(f"({i}) Artifact: {artifact.name}")

artifact_idx = int(input("Enter the index of the artifact to evaluate: "))
artifact = model_param_artifacts[artifact_idx]

params, _ = Segmentation.SAM.sam_lora_util.get_params_from_artifact(
    artifact, device)
sam_lora = Segmentation.SAM.sam_lora_util.initialize_sam_lora_with_params(
    run.config, params, device)
sam_lora.eval()


print(f"Loaded {len(params)} named parameters and {sum(p.numel() for p in params.values())} total parameters from finetuned model")
for name, param in params.items():
    print(f"\n--- {name} --- ")
    print(f"shape: {param.shape}")
    print(
        f"max: {param.max().item()}, min: {param.min().item()}, std: {param.std().item()}")

In [ ]:
def get_model_metrics(run) -> tuple:
    minloss_dice_score = run.summary.get(
        'test/params_minloss/mean_dice_score', None)
    final_dice_score = run.summary.get(
        'test/params_final/mean_dice_score', None)

    minloss_iou_score = run.summary.get(
        'test/params_minloss/mean_iou_score', None)
    final_iou_score = run.summary.get('test/params_final/mean_iou_score', None)

    minloss_tprec = run.summary.get('test/params_minloss/mean_tprec', None)
    final_tprec = run.summary.get('test/params_final/mean_tprec', None)

    minloss_tsens = run.summary.get('test/params_minloss/mean_tsens', None)
    final_tsens = run.summary.get('test/params_final/mean_tsens', None)

    minloss_clDice = run.summary.get(
        'test/params_minloss/mean_clDice_score', None)
    final_clDice = run.summary.get('test/params_final/mean_clDice_score', None)

    return (minloss_dice_score, final_dice_score,
            minloss_iou_score, final_iou_score,
            minloss_tprec, final_tprec,
            minloss_tsens, final_tsens,
            minloss_clDice, final_clDice)


(minloss_dice_score, final_dice_score,
 minloss_iou_score, final_iou_score,
 minloss_tprec, final_tprec,
 minloss_tsens, final_tsens,
 minloss_clDice, final_clDice) = get_model_metrics(run)

print("Saved model metrics on test set:\n====================")
print(f"minimal loss params dice score: {format_score(minloss_dice_score)}")
print(f"minimal loss params IoU score:  {format_score(minloss_iou_score)}")
print(f"minimal loss params tprec:      {format_score(minloss_tprec)}")
print(f"minimal loss params tsens:      {format_score(minloss_tsens)}")
print(f"minimal loss params clDice:     {format_score(minloss_clDice)}")
print(f"\nfinal params dice score: {format_score(final_dice_score)}")
print(f"final params IoU score:  {format_score(final_iou_score)}")
print(f"final params tprec:      {format_score(final_tprec)}")
print(f"final params tsens:      {format_score(final_tsens)}")
print(f"final params clDice:     {format_score(final_clDice)}")

In [ ]:

def show_mask(mask, ax, color: np.ndarray = None):
    if color is None:
        color = np.array([0.0, 1.0, 1.0, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)


def plot_images_masks_junctions(
    image: torch.Tensor,
    predicted_mask: np.ndarray,
    groundtruth_mask: np.ndarray,
    comparison_mask: np.ndarray = None,
    junction_coords: np.ndarray = None,
    skeleton: np.ndarray = None,
    ax=None,
    figsize=(10, 10),
    predicted_mask_color=None,
    groundtruth_mask_color=None,
    img_alpha=1.0,
    plot_grid: bool = True,
    prob_map: np.ndarray = None,
):
    if ax is None:
        plt.figure(figsize=figsize)
        ax = plt.gca()

    if image is not None:
        _, h, w = image.shape
        ax.imshow(image.permute(1, 2, 0).cpu().numpy(), alpha=img_alpha)
        if plot_grid:
            for i in range(1, 4):
                ax.axvline(x=i * (w / 4), color='red',
                           linestyle='-', linewidth=0.5)
                ax.axhline(y=i * (h / 4), color='red',
                           linestyle='-', linewidth=0.5)

    if predicted_mask is not None:
        show_mask(predicted_mask, ax, color=predicted_mask_color)
    if comparison_mask is not None:
        show_mask(comparison_mask, ax, color=np.array([1.0, 1.0, 0.0, 0.75]))
    if groundtruth_mask is not None:
        show_mask(groundtruth_mask, ax,
                  color=np.array([1.0, 0.0, 0.5, 0.5]) if groundtruth_mask_color is None
                  else groundtruth_mask_color)
    if prob_map is not None:
        im = ax.imshow(prob_map, cmap='hot', alpha=0.5, vmin=0, vmax=1)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Probability')

    if junction_coords is not None:
        ax.plot(junction_coords[:, 0], junction_coords[:, 1], 'ro',
                markersize=20, markerfacecolor='none', markeredgewidth=2)

    if skeleton is not None:
        skel_overlay = np.zeros((*skeleton.shape, 4), dtype=np.uint8)
        skel_overlay[skeleton > 0] = [255, 0, 0, 200]
        height, width = skeleton.shape
        ax.imshow(skel_overlay, extent=[0, width, height, 0], zorder=2)

    ax.axis('off')

In [ ]:
img_subdir = HIGHRES_IMG_PATCHES_DIR_NAME
mask_subdir = HIGHRES_MASK_PATCHES_DIR_NAME
patch_size = (1024, 1024)
test_img_dir = Path(DATASETS_DIR) / run.config["dataset"] / "test" / img_subdir
test_mask_dir = Path(DATASETS_DIR) / \
    run.config["dataset"] / "test" / mask_subdir

In [ ]:
full_img_names = Segmentation.PreProcessing.dataset_util.get_base_images(
    test_img_dir, exclude_soi_images=True)
full_img_names = sorted(full_img_names)

img_junction_coords = None
if DATASET_JUNCTION_COORDS_CVAT_XML_PATH is not None:
    img_junction_coords = Segmentation.PreProcessing.dataset_util.parse_junction_annotations_xml(
        DATASET_JUNCTION_COORDS_CVAT_XML_PATH)


def evaluate_output_masks(
    output_masks: torch.Tensor,
    groundtruth_mask_path,
    full_img_path,
    remove_small_objects: bool,
    ax=None,
    comparison_mask: torch.Tensor = None,
):
    postprocessed_mask, bboxes = Segmentation.PostProcessing.segmentation_postprocessing.postprocess_segmentation_masks(
        output_masks, grid_size=(4, 4), original_input_patch_img_size=patch_size,
        remove_small_objects=remove_small_objects)
    postprocessed_mask = postprocessed_mask.detach().cpu()

    original_img = load_transform_image(
        full_img_path, is_full_image=True).detach().cpu()
    groundtruth_mask = load_transform_image(
        groundtruth_mask_path, is_mask=True, is_full_image=True).detach().cpu()

    pred_junction_cords, pred_skeleton = None, None
    if remove_small_objects:
        pred_junction_cords, pred_skeleton = \
            JunctionDetection.SkeletonizeDetect.segmentation_junction_detection.detect_junctions_in_segmentation_mask(
                postprocessed_mask)

    comparison_mask_overlay = None
    if comparison_mask is not None:
        comparison_mask_overlay = (postprocessed_mask == 1) & (comparison_mask == 0)
    missed_groundtruth_mask = (groundtruth_mask == 1) & (postprocessed_mask == 0)

    hard_dice_score_pp = Segmentation.SAM.sam_lora_util.hard_dice_score(
        postprocessed_mask, groundtruth_mask)
    iou_score_pp = Segmentation.SAM.sam_lora_util.iou_score(
        postprocessed_mask, groundtruth_mask)

    output_mask_np = postprocessed_mask.squeeze(0).cpu().numpy()
    groundtruth_mask_np = groundtruth_mask.squeeze(0).cpu().numpy()
    clDice_pp, tprec_pp, tsens_pp = Segmentation.SAM.sam_lora_util.hard_clDice(
        output_mask_np, groundtruth_mask_np)

    print(f"\nEvaluation Results ({'small objects removed' if remove_small_objects else 'small objects NOT removed'}):\n====================")
    print(f"Number of Objects: {len(bboxes)}")
    print(f"Missed ground-truth pixels: {missed_groundtruth_mask.sum().item()}")
    print(f"Hard Dice Score: {format_score(hard_dice_score_pp)}")
    print(f"IoU Score:       {format_score(iou_score_pp)}")
    print(f"clDice / tprec / tsens: {format_score(clDice_pp)}, {format_score(tprec_pp)}, {format_score(tsens_pp)}")

    plot_images_masks_junctions(
        original_img, postprocessed_mask.numpy(), missed_groundtruth_mask.numpy(),
        comparison_mask=comparison_mask_overlay,
        junction_coords=pred_junction_cords, skeleton=pred_skeleton,
        ax=ax, figsize=(10, 10))

    return (hard_dice_score_pp, iou_score_pp, clDice_pp, tprec_pp, tsens_pp), \
        postprocessed_mask, original_img, groundtruth_mask


def evaluate_full_image(model: Segmentation.SAM.sam_lora.SamLoRA, full_img_name: str):
    patch_image_paths = sorted([
        path for path in test_img_dir.glob("*.png")
        if path.name.startswith(f"{full_img_name}_patch")
    ])

    full_img_path = Path(RAW_DATA_DIR) / HIGHRES_IMG_DIR_NAME / f"{full_img_name}.png"
    groundtruth_mask_path = Path(RAW_DATA_DIR) / HIGHRES_MASK_DIR_NAME / f"{full_img_name}.png"
    print(f"original image: {full_img_path}")

    output_masks, output_probs = [], []

    for img_path in patch_image_paths:
        img = load_transform_image(img_path, downsample_size=DOWNSAMPLE_SIZE)
        input_list = Segmentation.SAM.sam_lora_util.get_batched_input_list(
            img.unsqueeze(0).to(device))
        output = model(batched_input=input_list, multimask_output=False)
        output_masks.append(output[0]['masks'].squeeze(0))

        lowres_logits = output[0]['low_res_logits']
        resized_logits = model.sam_model.postprocess_masks(
            lowres_logits, input_size=(1024, 1024), original_size=(1024, 1024))
        output_probs.append(torch.sigmoid(resized_logits.squeeze(0)))

    output_masks = torch.stack(output_masks, dim=0).detach().cpu()
    output_probs = torch.stack(output_probs, dim=0).detach().cpu()

    _, axes = plt.subplots(1, 2, figsize=(14, 7))
    result_1, mask_smallobjremoved, original_img, groundtruth_mask = evaluate_output_masks(
        output_masks, groundtruth_mask_path, full_img_path,
        remove_small_objects=True, ax=axes[0])
    result_2, _, _, _ = evaluate_output_masks(
        output_masks, groundtruth_mask_path, full_img_path,
        remove_small_objects=False, ax=axes[1], comparison_mask=mask_smallobjremoved)
    plt.tight_layout()
    plt.show()
    plt.close()

    # Compute Betti curves on the stitched probability map
    output_probs_stitched = Segmentation.PostProcessing.segmentation_postprocessing.stitch_mask_tiles(
        output_probs, grid_size=(4, 4), original_input_patch_img_size=patch_size, as_uint=False)
    output_probs_np = output_probs_stitched.squeeze(0).detach().cpu().numpy()

    groundtruth_mask_np = groundtruth_mask.squeeze(0).cpu().numpy()
    groundtruth_b0, groundtruth_b1 = get_betti_at_thresholds(groundtruth_mask_np, [1.0])
    groundtruth_b0, groundtruth_b1 = groundtruth_b0[0], groundtruth_b1[0]
    print(f"Groundtruth Betti numbers: B0={groundtruth_b0}, B1={groundtruth_b1}")

    thresholds = np.linspace(0.0, 1.0, 11)
    pred_b0, pred_b1 = get_betti_at_thresholds(output_probs_np, thresholds)

    plot_betti_curve(thresholds, pred_b0, pred_b1, groundtruth_b0, groundtruth_b1)

    return result_1, result_2


hard_dice_scores_smallobjremoved, hard_dice_scores_inclsmallobj = [], []
iou_scores_smallobjremoved, iou_scores_inclsmallobj = [], []
clDice_scores_smallobjremoved, clDice_scores_inclsmallobj = [], []
tprecs_smallobjremoved, tprecs_inclsmallobj = [], []
tsenss_smallobjremoved, tsenss_inclsmallobj = [], []

for full_img_name in full_img_names:
    metrics_smallobjremoved, metrics_inclsmallobj = evaluate_full_image(sam_lora, full_img_name)

    hard_dice_scores_smallobjremoved.append(metrics_smallobjremoved[0])
    hard_dice_scores_inclsmallobj.append(metrics_inclsmallobj[0])
    iou_scores_smallobjremoved.append(metrics_smallobjremoved[1])
    iou_scores_inclsmallobj.append(metrics_inclsmallobj[1])
    clDice_scores_smallobjremoved.append(metrics_smallobjremoved[2])
    clDice_scores_inclsmallobj.append(metrics_inclsmallobj[2])
    tprecs_smallobjremoved.append(metrics_smallobjremoved[3])
    tprecs_inclsmallobj.append(metrics_inclsmallobj[3])
    tsenss_smallobjremoved.append(metrics_smallobjremoved[4])
    tsenss_inclsmallobj.append(metrics_inclsmallobj[4])

print("\nFull Image Evaluation Results (with vs. without small object removal)")
print(f"(removal threshold: {POSTPROCESSING_MIN_OBJ_SIZE} px)\n" + "="*50)
print(f"Mean Dice  (removed): {format_score(np.mean(hard_dice_scores_smallobjremoved))}")
print(f"Mean Dice  (kept):    {format_score(np.mean(hard_dice_scores_inclsmallobj))}")
print(f"Mean IoU   (removed): {format_score(np.mean(iou_scores_smallobjremoved))}")
print(f"Mean IoU   (kept):    {format_score(np.mean(iou_scores_inclsmallobj))}")
print(f"Mean clDice(removed): {format_score(np.mean(clDice_scores_smallobjremoved))}")
print(f"Mean clDice(kept):    {format_score(np.mean(clDice_scores_inclsmallobj))}")
print(f"Mean tprec (removed): {format_score(np.mean(tprecs_smallobjremoved))}")
print(f"Mean tprec (kept):    {format_score(np.mean(tprecs_inclsmallobj))}")
print(f"Mean tsens (removed): {format_score(np.mean(tsenss_smallobjremoved))}")
print(f"Mean tsens (kept):    {format_score(np.mean(tsenss_inclsmallobj))}")

In [ ]:
soi_img_names = Segmentation.PreProcessing.dataset_util.get_base_images(
    test_img_dir, exclude_soi_images=False)
soi_img_names = [n for n in soi_img_names if "_soi" in n]


def evaluate_soi_image(model: Segmentation.SAM.sam_lora.SamLoRA, soi_img_name: str):
    soi_patch_path = test_img_dir / f"{soi_img_name}_patch_00.png"
    groundtruth_mask_path = test_mask_dir / f"{soi_img_name}_patch_00.png"
    if not soi_patch_path.exists() or not groundtruth_mask_path.exists():
        print(f"SOI patch {soi_patch_path} or groundtruth {groundtruth_mask_path} not found, skipping.")
        return

    img = load_transform_image(soi_patch_path, downsample_size=DOWNSAMPLE_SIZE)
    input_list = Segmentation.SAM.sam_lora_util.get_batched_input_list(
        img.unsqueeze(0).to(device))
    output = model(batched_input=input_list, multimask_output=False)
    output_mask = output[0]['masks']
    cleaned_mask, = Segmentation.PostProcessing.segmentation_postprocessing.remove_small_objects_from_batch(
        output_mask).squeeze(0).detach().cpu()

    pred_junction_cords, pred_skeleton = \
        JunctionDetection.SkeletonizeDetect.segmentation_junction_detection.detect_junctions_in_segmentation_mask(
            cleaned_mask)

    comparison_mask = (output_mask.squeeze(0).detach().cpu() == 1) & (cleaned_mask == 0)

    groundtruth_mask = load_transform_image(
        groundtruth_mask_path, is_mask=True).detach().cpu()
    missed_groundtruth_mask = (groundtruth_mask == 1) & (cleaned_mask == 0)

    plot_images_masks_junctions(
        img, cleaned_mask.numpy(), missed_groundtruth_mask.numpy(),
        comparison_mask=comparison_mask.numpy(),
        junction_coords=pred_junction_cords, skeleton=pred_skeleton,
        figsize=(6, 6), plot_grid=False)


for soi_img_name in soi_img_names:
    evaluate_soi_image(sam_lora, soi_img_name)

### Visualize soft skeletonization (approximation) as used in clDice loss

In [ ]:
# soft_skeletonizer = Segmentation.SAM.sam_lora_util.SoftSkeletonize(num_iter=20)
#
# mask_path = test_mask_dir / f"{soi_img_names[1]}_patch_00.png"
# mask = load_transform_image(mask_path, is_mask=True).unsqueeze(0)
# soft_skeleton = soft_skeletonizer(mask).squeeze(0).cpu().numpy()
# plot_img_and_masks(image=None, predicted_mask=soft_skeleton, groundtruth_mask=mask.squeeze(
#    0).cpu().numpy(), figsize=(10, 10), predicted_mask_color=np.array([1.0, 0.0, 0.0, 1]))

### Plot Betti Curves 

In [ ]:
## --- 2. Computation ---
#thresholds = np.linspace(1.0, 0.0, 100)
#all_b0_curves, all_b1_curves = [], []
#
## --- 3. Plotting Individual Breakdowns (Grid) ---
#fig, axes = plt.subplots(3, num_imgs, figsize=(15, 12))
#
#for i in range(num_imgs):
#    diags = compute_persistence_diagrams(dummy_imgs[i])
#    b0_c, b1_c = get_betti_at_thresholds(diags, thresholds)
#    all_b0_curves.append(b0_c)
#    all_b1_curves.append(b1_c)
#
#
#plt.tight_layout()
#plt.show()
#
## --- 4. Aggregate Plot: F-Score and MAE ---
#all_b0_curves = np.array(all_b0_curves)
#all_b1_curves = np.array(all_b1_curves)
#f0_scores, mae1_scores = [], []
#
#for idx in range(len(thresholds)):
#    preds0 = all_b0_curves[:, idx]
#    batch_f1 = []
#    for p, gt in zip(preds0, gt_b0):
#        # Topological F-score calculation
#        if p == 0 and gt == 0:
#            batch_f1.append(1.0)
#        elif p == 0 or gt == 0:
#            batch_f1.append(0.0)
#        else:
#            rec = min(1.0, p / gt)
#            pre = min(1.0, gt / p)
#            batch_f1.append(2 * (pre * rec) / (pre + rec))
#    f0_scores.append(np.mean(batch_f1))
#
#    # MAE for Beta 1 (Holes)
#    mae1_scores.append(
#        np.mean(np.abs(all_b1_curves[:, idx] - np.array(gt_b1))))
#
#plt.figure(figsize=(10, 6))
#ax_f = plt.gca()
#ax_f.plot(thresholds, f0_scores, color='tab:blue',
#          linewidth=3, label='Connectivity F-Score')
#ax_f.set_xlabel('Probability Threshold')
#ax_f.set_ylabel('Avg. Beta0 F-Score', color='tab:blue', fontsize=12)
#ax_f.invert_xaxis()
#
#ax_e = ax_f.twinx()
#ax_e.plot(thresholds, mae1_scores, color='tab:red',
#          linestyle='--', linewidth=2, label='Hole Error (MAE)')
#ax_e.set_ylabel('Avg. Beta1 MAE', color='tab:red', fontsize=12)
#
#plt.title("Combined Topological Evaluation Profile", fontsize=14)
#ax_f.legend(loc='upper left')
#ax_e.legend(loc='upper right')
#plt.grid(alpha=0.3)
#plt.show()